In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import json
import time
from datetime import datetime
import tkinter as tk
from tkinter import messagebox
import threading  #To prevent the GUI from freezing

# ---- 1. Configuration ----
CALIB_IMAGES_DIR = "calibration_images"      
CALIB_PARAMS_FILE = "camera_calibration.json"
CHECKERBOARD = (8, 6)                        
SQUARE_SIZE_MM = 25.0                        

SCAN_LOG_CSV = "fabric_width_log.csv"
CAMERA_INDEX = 1

WHITE_THRESHOLD = 200        
WHITE_RATIO_STOP = 0.95      
FABRIC_THRESH = 127          

# 
CROP_TOP = 50       
CROP_BOTTOM = 50    
CROP_LEFT = 80      
CROP_RIGHT = 80     

os.makedirs(CALIB_IMAGES_DIR, exist_ok=True)
print("Crop Configuration Loaded.")

# ---- 2. Calibration Loader ----
def load_calibration():
    if os.path.exists(CALIB_PARAMS_FILE):
        with open(CALIB_PARAMS_FILE, "r") as f:
            data = json.load(f)
        return np.array(data["camera_matrix"]), np.array(data["dist_coeffs"]), data["pixel_to_mm_scale"]
    return None, None, 1.0

camera_matrix, dist_coeffs, pixel_to_mm_scale = load_calibration()

# ---- 3. Core Measurement Functions ----
def crop_frame(frame):
    h_frame, w_frame = frame.shape[:2]
    start_y = CROP_TOP
    end_y = h_frame - CROP_BOTTOM
    start_x = CROP_LEFT
    end_x = w_frame - CROP_RIGHT
    
    if start_y >= end_y or start_x >= end_x:
        return frame
        
    return frame[start_y:end_y, start_x:end_x]

def undistort_frame(frame, camera_matrix, dist_coeffs):
    if camera_matrix is None or dist_coeffs is None: return frame
    return cv2.undistort(frame, camera_matrix, dist_coeffs)

def is_white_background(frame, threshold=WHITE_THRESHOLD, white_ratio=WHITE_RATIO_STOP):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    return (cv2.countNonZero(thresh) / thresh.size) > white_ratio

def measure_width_px(frame, fabric_thresh=FABRIC_THRESH):
    h_frame, w_frame = frame.shape[:2]
    center_y = h_frame // 2  

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, fabric_thresh, 255, cv2.THRESH_BINARY_INV)

    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    row_pixels = thresh[center_y, :]
    fabric_indices = np.where(row_pixels == 255)[0]
    
    # Returns None if the fabric is not on the center line or if the measurement is 0.
    if len(fabric_indices) == 0: 
        return None, None

    x_start = int(np.min(fabric_indices))
    x_end = int(np.max(fabric_indices))
    width_px = x_end - x_start

    if width_px < 10: 
        return None, None

    visual_data = (x_start, x_end, center_y, w_frame)
    return width_px, visual_data

# ---- 4. Post-Scan Analysis & Plotting ----
def analyze_and_plot_scan(csv_path=SCAN_LOG_CSV):
    if not os.path.exists(csv_path): return
    df = pd.read_csv(csv_path)
    
    # Filter out zeros/invalid readings
    df_valid = df[df["width_mm"] > 0]
    if df_valid.empty: 
        print("No valid fabric measurements found.")
        return

    min_row = df_valid.loc[df_valid["width_mm"].idxmin()]
    summary = {
        "num_frames": len(df_valid),
        "min_width_mm": float(df_valid["width_mm"].min()),
        "min_width_frame": int(min_row["frame"]),
        "max_width_mm": float(df_valid["width_mm"].max()),
        "mean_width_mm": float(df_valid["width_mm"].mean()),
    }

    print("\n=== Scan Summary (Filtered Valid Measurements) ===")
    print(f"Minimum Valid Width : {summary['min_width_mm']:.2f} mm (Frame {summary['min_width_frame']})")
    print(f"Maximum Valid Width : {summary['max_width_mm']:.2f} mm")
    print(f"Mean Valid Width    : {summary['mean_width_mm']:.2f} mm")

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(df["frame"], df["width_mm"], color="#1f77b4", label="Width per frame (Includes Zeros)")
    ax.plot(df_valid["frame"], df_valid["width_mm"], color="#2ca02c", linestyle="None", marker=".", label="Valid Fabric Data")
    ax.axhline(summary["min_width_mm"], color="red", linestyle="--", label=f"True Min (Filtered) = {summary['min_width_mm']:.2f} mm")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("fabric_width_profile.png", dpi=150)
    plt.show(block=False)
    plt.pause(5)
    plt.close()

# ---- 5. Live Capture & Scan Loop ----
def run_scan(camera_index=CAMERA_INDEX, show_preview=True):
    global camera_matrix, dist_coeffs, pixel_to_mm_scale
    camera_matrix, dist_coeffs, pixel_to_mm_scale = load_calibration()

    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened(): 
        messagebox.showerror("Error", "Could not open camera.")
        return

    records = []
    frame_idx = 0
    print("Cropped Center-Line Scan Started...")

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame_u = undistort_frame(frame, camera_matrix, dist_coeffs)
        frame_cropped = crop_frame(frame_u)

        if is_white_background(frame_cropped): 
            print("White background detected. Stopping scan.")
            break

        width_px, visual_data = measure_width_px(frame_cropped)
        
        # Records 0.0 when the measurement is zero or invalid.
        if width_px is None:
            width_mm = 0.0
        else:
            width_mm = round(width_px * pixel_to_mm_scale, 2)
            
        records.append({
            "frame": frame_idx,
            "timestamp": datetime.now().isoformat(),
            "width_px": 0 if width_px is None else width_px,
            "width_mm": width_mm,
        })
            
        if show_preview and width_px is not None:
            x_start, x_end, center_y, w_frame = visual_data
            cv2.line(frame_cropped, (0, center_y), (w_frame, center_y), (255, 0, 0), 1)
            cv2.line(frame_cropped, (x_start, center_y), (x_end, center_y), (0, 255, 0), 3)
            cv2.circle(frame_cropped, (x_start, center_y), 4, (0, 0, 255), -1)
            cv2.circle(frame_cropped, (x_end, center_y), 4, (0, 0, 255), -1)
            cv2.putText(frame_cropped, f"Width: {width_mm:.1f} mm", (x_start, max(center_y - 15, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        elif show_preview:
            # Red warning line that shows when the fabric is not in the middle
            h_c, w_c = frame_cropped.shape[:2]
            cv2.line(frame_cropped, (0, h_c//2), (w_c, h_c//2), (0, 0, 255), 2)
            cv2.putText(frame_cropped, "NO FABRIC DETECTED IN CENTER", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        if show_preview:
            cv2.imshow("Cropped Fabric Scan", frame_cropped)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    if records:
        df = pd.DataFrame(records)
        df.to_csv(SCAN_LOG_CSV, index=False)
        analyze_and_plot_scan()

# ---- 6. GUI Setup (Threaded Control) ----
def start_scan_thread():
    # To prevent the GUI from freezing
    t = threading.Thread(target=run_scan)
    t.daemon = True
    t.start()

def launch_start_gui():
    root = tk.Tk()
    root.title("Fabric Roll Width Scanner")
    root.geometry("360x220")
    
    status_var = tk.StringVar(value="Cropped view mode active.\nInvalid center readings are excluded from Min Width.")

    tk.Label(root, text="Fabric Scanner", font=("Segoe UI", 11, "bold")).pack(pady=10)
    tk.Label(root, textvariable=status_var, fg="#333", wraplength=320, justify="center").pack(pady=5)
    
    tk.Button(root, text="Start Scan", command=start_scan_thread, bg="#2e7d32", fg="white", width=25, height=2).pack(pady=10)
    tk.Button(root, text="Close", command=root.destroy, width=15).pack(pady=5)

    root.mainloop()

if __name__ == "__main__":
    launch_start_gui()